# Phase 6: Imbalance Handling — Telco Customer Churn

## 1. Executive Summary & Problem Formulation
In Phase 5, all unadjusted baseline models demonstrated a critical **recall bottleneck** (~51%), missing nearly half of churned customers due to class imbalance (~73.5% retained vs. ~26.5% churned).

### Strategies Compared in Phase 6:
1. **Cost-Sensitive Learning (Algorithm-Level)**:
   - `class_weight='balanced'` for **Logistic Regression** and **Random Forest** (penalizes false negatives in proportion to class frequency).
   - `scale_pos_weight = 2.7686` for **XGBoost** (scales the gradient of positive instances by the negative-to-positive ratio $\frac{4,139}{1,495} \approx 2.77$).
2. **Resampling Techniques (Data-Level)**:
   - **SMOTE** (Synthetic Minority Over-sampling Technique) combined within an `imblearn.pipeline.Pipeline` to guarantee strict separation and eliminate data leakage.

### Evaluation Focus:
- Maximizing **Recall** while managing precision degradation and maintaining **ROC-AUC** / **PR-AUC** stability.

In [1]:
# 2. Environment Setup & Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML & Imbalance libraries
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

# Project modules
sys.path.append('..')
from src.feature_engineering import engineer_features, get_feature_types, build_preprocessor
from src.train import save_model
from src.evaluate import compare_models

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 3. Data Preparation & Class Imbalance Ratio Calculation

In [2]:
data_path = Path('../data/processed/cleaned_churn_data.csv')
df_clean = pd.read_csv(data_path)
df_feat = engineer_features(df_clean)

X = df_feat.drop(columns=['Churn'])
y = df_feat['Churn']
num_cols, cat_cols = get_feature_types(df_feat)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
imbalance_ratio = n_neg / n_pos
print(f"Training set size: {len(X_train)} samples")
print(f"  Retained (0): {n_neg:,} ({n_neg/len(X_train):.2%})")
print(f"  Churned  (1): {n_pos:,} ({n_pos/len(X_train):.2%})")
print(f"  Negative / Positive Imbalance Ratio: {imbalance_ratio:.4f}")

Training set size: 5634 samples
  Retained (0): 4,139 (73.46%)
  Churned  (1): 1,495 (26.54%)
  Negative / Positive Imbalance Ratio: 2.7686


## 4. Pipeline Setup: Baseline vs. Class Weights vs. SMOTE
We construct robust, leakage-free pipelines for each strategy.

In [3]:
pipelines = {
    # 1. Baselines
    'LogReg (Baseline)': Pipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'RandomForest (Baseline)': Pipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('clf', RandomForestClassifier(random_state=42))
    ]),
    'XGBoost (Baseline)': Pipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('clf', XGBClassifier(random_state=42, eval_metric='logloss'))
    ]),
    
    # 2. Cost-Sensitive / Class-Weighted
    'LogReg (Class-Weighted)': Pipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'RandomForest (Class-Weighted)': Pipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
    ]),
    'XGBoost (scale_pos_weight)': Pipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('clf', XGBClassifier(scale_pos_weight=imbalance_ratio, random_state=42, eval_metric='logloss'))
    ]),
    
    # 3. SMOTE Resampling
    'LogReg (SMOTE)': ImbPipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('smote', SMOTE(random_state=42)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'RandomForest (SMOTE)': ImbPipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('smote', SMOTE(random_state=42)),
        ('clf', RandomForestClassifier(random_state=42))
    ]),
    'XGBoost (SMOTE)': ImbPipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('smote', SMOTE(random_state=42)),
        ('clf', XGBClassifier(random_state=42, eval_metric='logloss'))
    ]),
}

fitted_models = {}
results = []

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fitted_models[name] = pipe
    
    results.append({
        'model': name,
        'accuracy': round(accuracy_score(y_test, y_pred), 4),
        'precision': round(precision_score(y_test, y_pred), 4),
        'recall': round(recall_score(y_test, y_pred), 4),
        'f1_score': round(f1_score(y_test, y_pred), 4),
        'roc_auc': round(roc_auc_score(y_test, y_prob), 4),
        'pr_auc': round(average_precision_score(y_test, y_prob), 4)
    })
    print(f"[x] Trained & Evaluated: {name}")

[x] Trained & Evaluated: LogReg (Baseline)
[x] Trained & Evaluated: RandomForest (Baseline)
[x] Trained & Evaluated: XGBoost (Baseline)
[x] Trained & Evaluated: LogReg (Class-Weighted)
[x] Trained & Evaluated: RandomForest (Class-Weighted)
[x] Trained & Evaluated: XGBoost (scale_pos_weight)
[x] Trained & Evaluated: LogReg (SMOTE)
[x] Trained & Evaluated: RandomForest (SMOTE)
[x] Trained & Evaluated: XGBoost (SMOTE)


## 5. Comparative Performance Analysis
Assessing the impact of class weighting and SMOTE on Recall, F1-Score, and overall discrimination.

In [4]:
res_df = pd.DataFrame(results).sort_values(by='recall', ascending=False).reset_index(drop=True)
print("=== IMBALANCE HANDLING EXPERIMENT COMPARISON ===")
print(res_df.to_string(index=False))

# Append to reports/model_results.csv
res_csv = Path('../reports/model_results.csv')
res_df.to_csv(res_csv, index=False)
print(f"\nUpdated results saved to {res_csv.resolve()}")

=== IMBALANCE HANDLING EXPERIMENT COMPARISON ===
                        model  accuracy  precision  recall  f1_score  roc_auc  pr_auc
               LogReg (SMOTE)    0.7381     0.5042  0.7968    0.6176   0.8408  0.6416
      LogReg (Class-Weighted)    0.7381     0.5043  0.7914    0.6160   0.8425  0.6400
   XGBoost (scale_pos_weight)    0.7594     0.5390  0.6471    0.5881   0.8195  0.6154
              XGBoost (SMOTE)    0.7757     0.5788  0.5695    0.5741   0.8251  0.6260
         RandomForest (SMOTE)    0.7757     0.5797  0.5642    0.5718   0.8312  0.6186
      RandomForest (Baseline)    0.7991     0.6532  0.5187    0.5782   0.8275  0.6285
            LogReg (Baseline)    0.7991     0.6542  0.5160    0.5770   0.8426  0.6394
           XGBoost (Baseline)    0.7864     0.6189  0.5080    0.5580   0.8276  0.6114
RandomForest (Class-Weighted)    0.7906     0.6376  0.4893    0.5537   0.8290  0.6247

Updated results saved to D:\ML\customer-churn-prediction\reports\model_results.csv


## 6. Confusion Matrix Shift Diagnostics
Examining how False Negatives (missed churners) are reduced by imbalance handling.

In [5]:
cm_baseline = confusion_matrix(y_test, fitted_models['LogReg (Baseline)'].predict(X_test))
cm_weighted = confusion_matrix(y_test, fitted_models['LogReg (Class-Weighted)'].predict(X_test))

print("--- Logistic Regression: Baseline vs. Class-Weighted ---")
print(f"Baseline Missed Churners (FN):        {cm_baseline[1, 0]} / {cm_baseline[1].sum()} ({cm_baseline[1, 0]/cm_baseline[1].sum():.2%})")
print(f"Class-Weighted Missed Churners (FN):   {cm_weighted[1, 0]} / {cm_weighted[1].sum()} ({cm_weighted[1, 0]/cm_weighted[1].sum():.2%})")
print(f"Reduction in Missed Churners:         {cm_baseline[1, 0] - cm_weighted[1, 0]} fewer churners missed! (+{(cm_baseline[1, 0] - cm_weighted[1, 0])/cm_baseline[1].sum():.2%} recall improvement)")

--- Logistic Regression: Baseline vs. Class-Weighted ---
Baseline Missed Churners (FN):        181 / 374 (48.40%)
Class-Weighted Missed Churners (FN):   78 / 374 (20.86%)
Reduction in Missed Churners:         103 fewer churners missed! (+27.54% recall improvement)


## 7. Model Persistence
Saving the top-performing balanced pipeline to `models/`.

In [6]:
best_balanced = fitted_models['LogReg (Class-Weighted)']
save_model(best_balanced, '../models/best_model.pkl')
save_model(best_balanced, '../models/logistic_regression.pkl')
print("[x] Saved models/best_model.pkl and models/logistic_regression.pkl with Class-Weighted Logistic Regression.")

[x] Saved models/best_model.pkl and models/logistic_regression.pkl with Class-Weighted Logistic Regression.


## 8. Strategic Conclusions for Phase 7

1. **Dramatic Recall Elevation**: Cost-sensitive weighting raises Logistic Regression recall from **51.60%** to **79.14%** (+27.54% points increase), and SMOTE achieves **79.68%**.
2. **Class-Weighting vs. SMOTE**: `class_weight='balanced'` achieves virtually identical recall and F1 to SMOTE, but with zero overhead, faster inference, and no risk of creating synthetic noise in borderline feature regions.
3. **XGBoost Sensitivity**: Setting `scale_pos_weight = 2.77` in XGBoost increases recall from **50.80%** to **64.71%**, demonstrating solid responsiveness to minority weight adjustments.
4. **Handoff to Phase 7**: In Phase 7 (Model Comparison & Tuning), we will systematically tune hyperparameters for both **Logistic Regression** and **XGBoost** with balanced weighting to establish the production candidate.